# Site Road Feature Extraction

Enriches each **traffic-count station** (`sites.csv`) with:

| Feature group | Source |
|---|---|
| Road name, morphology, category, access, length | Wegenregister (`wegsegment_clean.gpkg`) |
| Bike-lane width on nearest OSM way | OpenStreetMap via OSMnx |
| Distance to nearest city centre | OSM place nodes |
| POI counts at 250 m / 500 m / 1 km radii | OSM amenity tags |

**Output:** `explo/andy/sites_enriched.csv` and `explo/andy/sites_enriched.gpkg`


## 1  Imports & Settings


In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
import osmnx as ox
import warnings, os, time
from shapely.geometry import Point
from shapely.ops import nearest_points

warnings.filterwarnings('ignore')
ox.settings.log_console = False
ox.settings.use_cache   = True          # cache OSM downloads

# ── Paths ────────────────────────────────────────────────────────────────────
SITES_CSV      = '../../data/sites.csv'
WEGEN_GPKG     = 'explo/andy/wegsegment_clean.gpkg'
OUTPUT_DIR     = 'explo/andy'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── POI radii (metres) ───────────────────────────────────────────────────────
POI_RADII = [250, 500, 1000]

# ── POI categories to count (OSM amenity / tag values) ──────────────────────
POI_TAGS = {
    'shop'       : {'shop': True},
    'education'  : {'amenity': ['school', 'university', 'college', 'kindergarten']},
    'hotel'      : {'tourism': ['hotel', 'hostel', 'motel', 'guest_house']},
    'hospital'   : {'amenity': ['hospital', 'clinic', 'doctors', 'pharmacy']},
}

print('Libraries loaded.')


Libraries loaded.


## 2  Load & Inspect Traffic-Count Sites


In [2]:
SITE_COLS = [
    'row_id', 'site_id', 'longitude', 'latitude',
    'site_name', 'operator', 'road_code', 'district_code',
    'municipality', 'interval_min', 'install_date'
]

sites = pd.read_csv(SITES_CSV, header=None, names=SITE_COLS)
sites['install_date'] = pd.to_datetime(sites['install_date'], errors='coerce')

print(f'Sites loaded: {len(sites):,} rows')
display(sites.head(5))
print('\nMissing values:')
print(sites.isna().sum()[sites.isna().sum() > 0])


Sites loaded: 151 rows


,row_id,site_id,longitude,latitude,site_name,operator,road_code,district_code,municipality,interval_min,install_date
0,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22
1,2,100052862,4.471690,51.275120,Brasschaat 2,Vlaamse Overheid A. Wegen enVerkeer,N0010002,AWV123,Brasschaat,15,2019-08-22
2,3,100052863,4.472220,51.275030,Brasschaat 1,Vlaamse Overheid A. Wegen enVerkeer,N0010001,AWV123,Brasschaat,15,2019-08-22
3,4,100052864,5.190110,51.160230,Balen 1,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,2019-08-22
4,5,100052865,5.190030,51.160180,Balen 2,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,2019-08-22



Missing values:
road_code        3
district_code    3
municipality     1
dtype: int64


## 3  Convert to GeoDataFrame (WGS84 → Lambert 72)


In [3]:
# Build GeoDataFrame in WGS84 (EPSG:4326) – needed for OSM queries
sites_wgs84 = gpd.GeoDataFrame(
    sites,
    geometry=gpd.points_from_xy(sites['longitude'], sites['latitude']),
    crs='EPSG:4326'
)

# Also project to Lambert 72 (EPSG:31370) – same CRS as Wegenregister,
# and needed for accurate distance calculations in metres
sites_lam = sites_wgs84.to_crs('EPSG:31370')

print('WGS84 GeoDataFrame  :', sites_wgs84.crs)
print('Lambert 72 GeoDataFrame:', sites_lam.crs)
sites_lam.head(3)


WGS84 GeoDataFrame  : EPSG:4326
Lambert 72 GeoDataFrame: EPSG:31370


,row_id,site_id,longitude,latitude,site_name,operator,road_code,district_code,municipality,interval_min,install_date,geometry
0,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22,POINT (156143.904 178432.685)
1,2,100052862,4.471690,51.275120,Brasschaat 2,Vlaamse Overheid A. Wegen enVerkeer,N0010002,AWV123,Brasschaat,15,2019-08-22,POINT (157182.957 218365.646)
2,3,100052863,4.472220,51.275030,Brasschaat 1,Vlaamse Overheid A. Wegen enVerkeer,N0010001,AWV123,Brasschaat,15,2019-08-22,POINT (157219.957 218355.684)


## 4  Enrich with Wegenregister Infrastructure Features

Strategy: **nearest-segment spatial join** – for each site we find the
closest road segment in the cleaned Wegenregister GeoPackage and attach
its attributes.


In [4]:
print('Loading Wegenregister GeoPackage...')
t0 = time.time()
wegen = gpd.read_file(WEGEN_GPKG)
print(f'  {len(wegen):,} segments loaded in {time.time()-t0:.1f} s')
print('  Columns:', wegen.columns.tolist())


Loading Wegenregister GeoPackage...
  834,974 segments loaded in 14.2 s
  Columns: ['segment_id', 'start_node_id', 'end_node_id', 'status_en', 'morphology_code', 'morphology_en', 'road_category_code', 'road_category_en', 'access_code', 'access_en', 'geometry_method_en', 'left_streetname', 'right_streetname', 'manager_code', 'manager_label', 'record_date', 'length_m', 'geometry']


In [6]:
# # ── Nearest-segment join ────────────────────────────────────────────────────
# # gpd.sjoin_nearest (requires geopandas >= 0.10)
# WEGEN_KEEP = [
#     'segment_id', 'morphology_en', 'road_category_code', 'road_category_en',
#     'access_en', 'geometry_method_en', 'left_streetname', 'right_streetname',
#     'manager_code', 'manager_label', 'length_m', 'geometry'
# ]
# wegen_sub = wegen[[c for c in WEGEN_KEEP if c in wegen.columns]].copy()

# print('Running nearest-segment spatial join...')
# t0 = time.time()
# sites_joined = gpd.sjoin_nearest(
#     sites_lam,
#     wegen_sub,
#     how='left',
#     distance_col='dist_to_segment_m',
#     lsuffix='site',
#     rsuffix='wegen'
# )
# # Drop duplicate rows that can arise from equidistant segments
# sites_joined = sites_joined.drop_duplicates(subset='site_id').copy()
# print(f'  Done in {time.time()-t0:.1f} s – shape: {sites_joined.shape}')

# # Derive a single representative road name from left/right street name
# sites_joined['road_name'] = sites_joined['left_streetname'].where(
#     sites_joined['left_streetname'].notna(),
#     sites_joined['right_streetname']
# )

# print('\nSample Wegenregister columns attached:')
# display(sites_joined[['site_id','site_name','road_name','morphology_en',
#                        'road_category_en','access_en','dist_to_segment_m']].head(8))


# ── [NEW] Load and merge the Wegenregister width data ───────────────────────
print('Loading official Wegenregister widths from AttWegbreedte.dbf...')
width_dbf_path = "../../data/extra/Wegenregister/Shapefile/AttWegbreedte.dbf" 
width_df = gpd.read_file(width_dbf_path)

# Clean domain codes -8 (unknown) and -9 (not applicable) to NaN
width_df['BREEDTE'] = width_df['BREEDTE'].replace([-8, -9], np.nan)
print("  Cleaned width data (replaced -8, -9 with NaN).")


# ── 2. Merge widths into the main 'wegen' dataframe ──────────────────────
# Make the cell safe to run multiple times by dropping the column if it exists
if 'official_road_width_m' in wegen.columns:
    wegen = wegen.drop(columns=['official_road_width_m'])
if 'WS_OIDN' in wegen.columns:
    wegen = wegen.drop(columns=['WS_OIDN'])

# Merge based on the unique ID
wegen = wegen.merge(
    width_df[['WS_OIDN', 'BREEDTE']], 
    left_on='segment_id',   # Check if your ID column is 'segment_id' or 'segment_object_id'
    right_on='WS_OIDN', 
    how='left'
)

# Rename the column to English
wegen.rename(columns={'BREEDTE': 'official_road_width_m'}, inplace=True)

# Now we can safely calculate the valid count
valid_count = wegen['official_road_width_m'].notna().sum()
print(f"  Widths successfully merged. Found {valid_count:,} valid width records.")


# ── 3. Nearest-segment join (Sensor Sites to Roads) ──────────────────────
# gpd.sjoin_nearest (requires geopandas >= 0.10)
WEGEN_KEEP = [
    'segment_id', 'morphology_en', 'road_category_code', 'road_category_en',
    'access_en', 'geometry_method_en', 'left_streetname', 'right_streetname',
    'manager_code', 'manager_label', 'length_m', 'official_road_width_m', 'geometry'
]

# Ensure we only keep columns that actually exist to prevent KeyErrors
wegen_sub = wegen[[c for c in WEGEN_KEEP if c in wegen.columns]].copy()

print('\nRunning nearest-segment spatial join...')
t0 = time.time()
sites_joined = gpd.sjoin_nearest(
    sites_lam,
    wegen_sub,
    how='left',
    distance_col='dist_to_segment_m',
    lsuffix='site',
    rsuffix='wegen'
)

# Drop duplicate rows that can arise from equidistant segments
sites_joined = sites_joined.drop_duplicates(subset='site_id').copy()
print(f'  Done in {time.time()-t0:.1f} s – shape: {sites_joined.shape}')


# ── 4. Final Polish ──────────────────────────────────────────────────────
# Derive a single representative road name from left/right street name
sites_joined['road_name'] = sites_joined['left_streetname'].where(
    sites_joined['left_streetname'].notna(),
    sites_joined['right_streetname']
)

print('\nSample Wegenregister columns attached:')
display_cols = [
    'site_id', 'site_name', 'road_name', 'morphology_en',
    'road_category_en', 'access_en', 'dist_to_segment_m', 'official_road_width_m'
]
# Safely display only the columns that successfully joined
display(sites_joined[[c for c in display_cols if c in sites_joined.columns]].head(8))

Loading official Wegenregister widths from AttWegbreedte.dbf...
  Cleaned width data (replaced -8, -9 with NaN).
  Widths successfully merged. Found 609,468 valid width records.

Running nearest-segment spatial join...
  Done in 0.8 s – shape: (151, 26)

Sample Wegenregister columns attached:


,site_id,site_name,road_name,morphology_en,road_category_en,access_en,dist_to_segment_m,official_road_width_m
0,100046096,Machelen,None,cycleway_footpath,not_applicable,public_road,2.878127,NaN
1,100052862,Brasschaat 2,Bredabaan,divided_road_non_motorway,None,public_road,7.577427,7.0
2,100052863,Brasschaat 1,Bredabaan,divided_road_non_motorway,None,public_road,12.860851,7.0
3,100052864,Balen 1,Schoor,single_carriageway,None,public_road,3.525611,6.0
4,100052865,Balen 2,Schoor,single_carriageway,None,public_road,4.312523,6.0
5,100052866,Evergem 2,Christoffelweg,single_carriageway,None,public_road,2.992821,6.0
6,100052867,Evergem 1,Christoffelweg,single_carriageway,None,public_road,8.067687,6.0
7,100052868,Heist op den Berg 1,Mechelsesteenweg,single_carriageway,None,public_road,6.232753,7.0


## 5  Bike-Lane Width from OpenStreetMap

### Why most sites return NaN — 3 root causes

| # | Root cause | Detail |
|---|---|---|
| 1 | **`cycleway:width` is almost never tagged in OSM** | This tag requires a surveyor to measure and record the exact width. Globally < 0.1 % of ways have it; in Belgium it is essentially absent. |
| 2 | **Traffic-count stations sit on vehicle roads, not cycle paths** | The sites are on regional/national roads (N-roads, R-roads). OSM tags the *road* (`highway=secondary`) but rarely adds a numeric `cycleway:width` sub-tag even when a painted lane exists. |
| 3 | **Search radius was too small (100 m)** | Rural sites may have the nearest named road > 100 m away, so the query returned zero features. |

### Fix — 4-tier priority chain + wider radius

The new function uses a **200 m radius** and tries tags in this priority order:

1. `cycleway:width` / `cycleway:right:width` / `cycleway:left:width` / `cycleway:both:width` — explicit width (rare but most accurate)
2. `width` on ways that **also carry a `cycleway` tag** — road-level width where a cycleway is confirmed
3. `width` on dedicated cycle-infrastructure ways (`highway=cycleway/path/footway`) — the way *is* the bike lane
4. **`has_cycleway`** boolean flag — records whether any cycleway presence tag exists, even if no width is available


In [13]:
sites_joined[sites_joined['official_road_width_m'] == 4]['morphology_en'].value_counts()

morphology_en
divided_road_non_motorway    4
single_carriageway           1
cycleway_footpath            1
ramp_grade_separated         1
Name: count, dtype: int64

In [15]:
WIDTH_TAGS_PRIORITY = [
    'cycleway:width',
    'cycleway:right:width',
    'cycleway:left:width',
    'cycleway:both:width',
    'cycleway:lane:width',
]
BIKE_HIGHWAY_TYPES = {'cycleway', 'path', 'footway', 'bridleway', 'track'}
CYCLEWAY_PRESENCE_TAGS = [
    'cycleway', 'cycleway:right', 'cycleway:left', 'cycleway:both',
    'cycleway:lane', 'cyclestreet', 'oneway:bicycle',
]


def _parse_width(val):
    """Parse width strings like '2.5', '2.5 m', '250 cm' into float metres."""
    if pd.isna(val):
        return np.nan
    s = str(val).lower().strip()
    if s.endswith('cm'):
        try:
            return float(s.replace('cm', '').strip()) / 100
        except ValueError:
            pass
    for unit in [' meters', ' meter', ' m', 'm']:
        s = s.replace(unit, '').strip()
    try:
        return float(s)
    except ValueError:
        return np.nan


def get_bike_info(lat, lon, official_width=np.nan, dist=200):
    result = {'bike_lane_width_m': np.nan, 'bike_lane_source': 'missing:no_osm_features', 'has_cycleway': False}
    try:
        gdf = ox.features_from_point((lat, lon), tags={'highway': True}, dist=dist)
        if gdf.empty:
            # If OSM is completely empty here, check Wegenregister before giving up!
            if pd.notna(official_width):
                result.update({'bike_lane_width_m': float(official_width), 'bike_lane_source': 'tier4:wegenregister_official_width'})
            return result

        # --- PRE-CHECK: Does a cycleway exist here? ---
        rows_with_cycleway = pd.Series(False, index=gdf.index)
        for c in CYCLEWAY_PRESENCE_TAGS:
            if c in gdf.columns:
                valid = gdf[c].dropna()
                positive = valid[~valid.astype(str).str.lower().isin(['no', 'none', 'nan'])]
                rows_with_cycleway |= gdf.index.isin(positive.index)
        
        if rows_with_cycleway.any() or (
            'highway' in gdf.columns and gdf['highway'].isin(BIKE_HIGHWAY_TYPES).any()
        ):
            result['has_cycleway'] = True

        # --- TIER 1: Explicit Bike Lane Width ---
        for tag in WIDTH_TAGS_PRIORITY:
            if tag in gdf.columns:
                vals = gdf[tag].dropna()
                if not vals.empty:
                    w = _parse_width(vals.iloc[0])
                    if not np.isnan(w):
                        result.update({
                            'bike_lane_width_m': w, 
                            'bike_lane_source': 'tier1:explicit_cycleway_width', 
                            'has_cycleway': True
                        })
                        return result

        # --- TIER 2: Dedicated Bike/Pedestrian Path Width ---
        if 'highway' in gdf.columns and 'width' in gdf.columns:
            bike_ways = gdf[gdf['highway'].isin(BIKE_HIGHWAY_TYPES)]
            widths = bike_ways['width'].dropna() if not bike_ways.empty else pd.Series(dtype=float)
            if not widths.empty:
                w = _parse_width(widths.iloc[0])
                if not np.isnan(w):
                    result.update({
                        'bike_lane_width_m': w, 
                        'bike_lane_source': 'tier2:dedicated_path_width', 
                        'has_cycleway': True
                    })
                    return result

        # --- TIER 3: General Road/Street Width (from OSM) ---
        if 'width' in gdf.columns:
            general_widths = gdf['width'].dropna()
            if not general_widths.empty:
                w = _parse_width(general_widths.iloc[0])
                if not np.isnan(w):
                    source_label = 'tier3:shared_road_width_with_cycleway' if result['has_cycleway'] else 'tier3:general_road_width'
                    result.update({
                        'bike_lane_width_m': w, 
                        'bike_lane_source': source_label
                    })
                    return result

        # --- TIER 4: Official Government Data (Wegenregister Fallback) ---
        # If OSM didn't have any width data, we use the official width we joined earlier
        if pd.notna(official_width):
            source_label = 'tier4:wegenregister_with_cycleway' if result['has_cycleway'] else 'tier4:wegenregister_general_road'
            result.update({
                'bike_lane_width_m': float(official_width),
                'bike_lane_source': source_label
            })
            return result

        # --- MISSING FALLBACKS ---
        # No width data was found in OSM, and the Wegenregister was also missing it
        result['bike_lane_source'] = (
            'missing:has_cycleway_no_width' if result['has_cycleway'] else 'missing:no_width_data'
        )
        
    except Exception as e:
        result['bike_lane_source'] = f'error:{type(e).__name__}'
        
    return result


BIKE_CACHE = os.path.join(OUTPUT_DIR, 'bike_cache.csv')

if os.path.exists(BIKE_CACHE):
    bike_df = pd.read_csv(BIKE_CACHE)
    sites_joined['bike_lane_width_m'] = bike_df['bike_lane_width_m'].values
    sites_joined['bike_lane_source']  = bike_df['bike_lane_source'].values
    sites_joined['has_cycleway']      = bike_df['has_cycleway'].values
    print(f'Loaded bike-lane info from cache: {BIKE_CACHE}')
else:
    print('Extracting bike-lane info (200 m radius, 4-tier fallback)...')
    t0 = time.time()
    
    # [MODIFIED] Pass the official width into the function
    bike_results = [
        get_bike_info(r['latitude'], r['longitude'], official_width=r.get('official_road_width_m', np.nan))
        for _, r in sites_joined.iterrows()
    ]
    
    sites_joined['bike_lane_width_m'] = [d['bike_lane_width_m'] for d in bike_results]
    sites_joined['bike_lane_source']  = [d['bike_lane_source']  for d in bike_results]
    sites_joined['has_cycleway']      = [d['has_cycleway']      for d in bike_results]
    
    # Save cache
    sites_joined[['site_id', 'bike_lane_width_m', 'bike_lane_source', 'has_cycleway']].to_csv(BIKE_CACHE, index=False)
    print(f'Done in {time.time()-t0:.1f} s — saved to {BIKE_CACHE}')

n_total = len(sites_joined)
n_width = sites_joined['bike_lane_width_m'].notna().sum()
n_cyc   = sites_joined['has_cycleway'].sum()
print(f'  Width extracted : {n_width}/{n_total} sites ({n_width/n_total*100:.1f} %)')
print(f'  Has cycleway tag: {n_cyc}/{n_total} sites ({n_cyc/n_total*100:.1f} %)')
print(sites_joined['bike_lane_source'].value_counts().to_string())


Extracting bike-lane info (200 m radius, 4-tier fallback)...
Done in 14.2 s — saved to explo/andy\bike_cache.csv
  Width extracted : 143/151 sites (94.7 %)
  Has cycleway tag: 151/151 sites (100.0 %)
bike_lane_source
tier2:dedicated_path_width               62
tier4:wegenregister_with_cycleway        62
tier1:explicit_cycleway_width            10
tier3:shared_road_width_with_cycleway     9
missing:has_cycleway_no_width             8


In [18]:
sites_joined.head()

,row_id,site_id,longitude,latitude,site_name,operator,road_code,district_code,municipality,interval_min,...,left_streetname,right_streetname,manager_code,manager_label,length_m,dist_to_segment_m,road_name,bike_lane_width_m,bike_lane_source,has_cycleway
0,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,...,None,None,AWV212,District Vilvoorde-gewestwegen,847.57,2.878127,None,4.0,tier3:bike_highway_width,True
1,2,100052862,4.471690,51.275120,Brasschaat 2,Vlaamse Overheid A. Wegen enVerkeer,N0010002,AWV123,Brasschaat,15,...,Bredabaan,Bredabaan,AWV123,District Brecht,398.76,7.577427,Bredabaan,NaN,tier4:no_cycleway_tag,False
2,3,100052863,4.472220,51.275030,Brasschaat 1,Vlaamse Overheid A. Wegen enVerkeer,N0010001,AWV123,Brasschaat,15,...,Bredabaan,Bredabaan,AWV123,District Brecht,288.86,12.860851,Bredabaan,NaN,tier4:no_cycleway_tag,False
3,4,100052864,5.190110,51.160230,Balen 1,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,...,Schoor,Schoor,AWV114,District Geel,254.23,3.525611,Schoor,NaN,tier4:no_cycleway_tag,False
4,5,100052865,5.190030,51.160180,Balen 2,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,...,Schoor,Schoor,AWV114,District Geel,254.23,4.312523,Schoor,NaN,tier4:no_cycleway_tag,False


## 6  Distance to Nearest City Centre (OSM)

### Why the old approach was slow (69+ minutes)

| Problem | Detail |
|---------|--------|
| One Overpass API request **per site** | 151 HTTP round-trips, each taking 20–30 s |
| `search_dist=50_000` (50 km radius) | Each request downloaded every city/town in a huge area |
| No result sharing | Two sites 2 km apart re-downloaded the same data from scratch |

### Fix — download once, compute all distances locally

1. Fetch **all** Belgian city/town nodes **once** with a single bounding-box query
2. Vectorise the Haversine computation with **numpy broadcasting** (no Python loop)
3. Result: **1 API call** instead of 151 → runs in seconds


In [16]:
import osmnx as ox
import geopandas as gpd
import time

# 1. FIXED: Bounding Box is now (West, South, East, North)
BELGIUM_BBOX = (2.50, 49.45, 6.45, 51.55) 
CITIES_CACHE = 'cities_cache.gpkg' # Update path as needed

# --- Fetch or Load Data ---
if os.path.exists(CITIES_CACHE):
    cities_pts = gpd.read_file(CITIES_CACHE)
    print(f'Loaded {len(cities_pts):,} Belgian cities from cache.')
else:
    print('Fetching all Belgian city/town nodes from OSM...')
    # Pass the fixed tuple directly
    cities_gdf = ox.features_from_bbox(bbox=BELGIUM_BBOX, tags={'place': ['city', 'town']})
    
    # Filter for Points and handle missing names
    cities_pts = cities_gdf[cities_gdf.geometry.geom_type == 'Point'].copy()
    cities_pts['name'] = cities_pts['name'].fillna("Unknown")
    
    cities_pts[['name', 'geometry']].to_file(CITIES_CACHE, driver='GPKG')
    print(f'Cached {len(cities_pts):,} Belgian city nodes.')

# --- The "GeoPandas Way" to Calculate Distance ---
print('Computing nearest city for each site...')

# Ensure both dataframes are using the Belgian metric CRS (Lambert 72)
cities_pts = cities_pts.to_crs("EPSG:31370")

# Assuming your 'sites_joined' is already a GeoDataFrame. If not, make it one:
# sites_joined = gpd.GeoDataFrame(sites_joined, geometry=gpd.points_from_xy(sites_joined.longitude, sites_joined.latitude), crs="EPSG:4326")
if sites_joined.crs != "EPSG:31370":
    sites_joined = sites_joined.to_crs("EPSG:31370")

# Perform a spatial nearest join. This automatically finds the closest city point 
# and calculates the exact distance in meters.
sites_with_city = gpd.sjoin_nearest(
    sites_joined, 
    cities_pts[['name', 'geometry']], 
    how='left', 
    distance_col='dist_to_city_m'
)

# Rename the merged OSM 'name' column to your preferred naming convention
sites_with_city.rename(columns={'name': 'nearest_city'}, inplace=True)

print(sites_with_city[['site_name', 'nearest_city', 'dist_to_city_m']].head())

Loaded 563 Belgian cities from cache.
Computing nearest city for each site...
      site_name nearest_city  dist_to_city_m
0      Machelen     Machelen     1679.177922
1  Brasschaat 2   Brasschaat     2172.854627
2  Brasschaat 1   Brasschaat     2157.003995
3       Balen 1        Balen     1847.183374
4       Balen 2        Balen     1846.398929


In [20]:
sites_with_city.head()

,row_id,site_id,longitude,latitude,site_name,operator,road_code,district_code,municipality,interval_min,...,length_m,dist_to_segment_m,road_name,bike_lane_width_m,bike_lane_source,has_cycleway,element,id,nearest_city,dist_to_city_m
0,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,...,847.57,2.878127,None,4.0,tier3:bike_highway_width,True,node,308897364,Machelen,1679.177922
1,2,100052862,4.471690,51.275120,Brasschaat 2,Vlaamse Overheid A. Wegen enVerkeer,N0010002,AWV123,Brasschaat,15,...,398.76,7.577427,Bredabaan,NaN,tier4:no_cycleway_tag,False,node,249698703,Brasschaat,2172.854627
2,3,100052863,4.472220,51.275030,Brasschaat 1,Vlaamse Overheid A. Wegen enVerkeer,N0010001,AWV123,Brasschaat,15,...,288.86,12.860851,Bredabaan,NaN,tier4:no_cycleway_tag,False,node,249698703,Brasschaat,2157.003995
3,4,100052864,5.190110,51.160230,Balen 1,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,...,254.23,3.525611,Schoor,NaN,tier4:no_cycleway_tag,False,node,2573491654,Balen,1847.183374
4,5,100052865,5.190030,51.160180,Balen 2,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,...,254.23,4.312523,Schoor,NaN,tier4:no_cycleway_tag,False,node,2573491654,Balen,1846.398929


## 7  POI Counts at Multiple Radii (OSM)

### Why the naive approach would be extremely slow

The original `count_pois()` called `ox.features_from_point()` for **every site × category × radius**:
151 sites × 4 categories × 3 radii = **1,812 API calls** → estimated **10+ hours**.

### Fix — download each category once, filter locally

1. For each POI category, fetch **all matching features in Belgium** with one bounding-box query (4 queries total)
2. Convert sites to a projected CRS (Lambert 72, unit = metres)
3. Use `gdf.within(site.buffer(radius))` — pure local GeoPandas, no API calls
4. Result: **4 API calls** instead of 1,812 → runs in seconds


In [18]:


# Assuming POI_TAGS, POI_RADII, BELGIUM_BBOX, and sites_joined are defined

# ── Step 1: Download ALL Belgian POIs — one request per category ─────────────
poi_gdfs = {}   # category -> GeoDataFrame projected to Lambert 72

print('Loading or Downloading POI data for Belgium...')
for cat, tags in POI_TAGS.items():
    t0 = time.time()
    
    # Define a specific cache file for each POI category
    cache_path = os.path.join(OUTPUT_DIR, f'poi_cache_{cat}.gpkg')
    
    # --- Check if the cache exists ---
    if os.path.exists(cache_path):
        # Load directly from the local GeoPackage
        gdf = gpd.read_file(cache_path)
        poi_gdfs[cat] = gdf
        print(f'  {cat:12s}: {len(gdf):,} features LOADED FROM CACHE in {time.time()-t0:.1f} s')
        
    else:
        # --- Cache missing: Fetch from OSM ---
        try:
            gdf = ox.features_from_bbox(bbox=BELGIUM_BBOX, tags=tags)
            
            # 1. Project to Lambert 72 (meters) FIRST
            gdf = gdf.to_crs('EPSG:31370')
            
            # 2. Calculate the centroid on the flat metric projection
            gdf = gdf.copy()
            gdf['geometry'] = gdf.geometry.centroid
            
            # 3. Save to Cache (Geometry ONLY to prevent list-column crash errors)
            # We create a temporary GeoDataFrame with just the geometry column
            cache_gdf = gpd.GeoDataFrame(geometry=gdf.geometry, crs='EPSG:31370')
            cache_gdf.to_file(cache_path, driver='GPKG')
            
            poi_gdfs[cat] = gdf
            print(f'  {cat:12s}: {len(gdf):,} features DOWNLOADED & CACHED in {time.time()-t0:.1f} s')
            
        except Exception as e:
            print(f'  {cat:12s}: no features found ({type(e).__name__}), filling with 0')
            poi_gdfs[cat] = None

print()

# ── Step 2: Count POIs per radius using blazing-fast Spatial Joins (sjoin) ───
print('Counting POIs per site per radius (Optimized sjoin)...')
t0 = time.time()

# Ensure sites_joined is in Lambert 72 and has a clean integer index to group by
if sites_joined.crs != 'EPSG:31370':
    sites_joined = sites_joined.to_crs('EPSG:31370')
sites_joined = sites_joined.reset_index(drop=True)

for cat, poi_gdf in poi_gdfs.items():
    if poi_gdf is None or poi_gdf.empty:
        for r in POI_RADII:
            sites_joined[f'poi_{cat}_{r}m'] = 0
        continue

    for r in POI_RADII:
        col = f'poi_{cat}_{r}m'
        
        # Create a temporary GeoDataFrame of the buffered sites (circles)
        buffered_sites = sites_joined[['geometry']].copy()
        buffered_sites['geometry'] = buffered_sites.geometry.buffer(r)
        
        # Perform an inner spatial join: Which POIs fall inside which site buffers?
        # predicate='within' ensures we only match POIs strictly inside the buffer
        joined = gpd.sjoin(poi_gdf, buffered_sites, how='inner', predicate='within')
        
        # Count the number of POIs per site index
        # 'index_right' contains the index of the site from buffered_sites
        counts = joined.groupby('index_right').size()
        
        # Map the counts back to the main dataframe, filling NaNs with 0
        sites_joined[col] = sites_joined.index.map(counts).fillna(0).astype(int)
        
        print(f'  {col}: done  (sum={sites_joined[col].sum():,})')

print(f'\nAll POI counts done in {time.time()-t0:.1f} s')

Loading or Downloading POI data for Belgium...
  shop        : 96,176 features DOWNLOADED & CACHED in 40.5 s
  education   : 17,077 features DOWNLOADED & CACHED in 5.6 s
  hotel       : 5,269 features DOWNLOADED & CACHED in 2.3 s
  hospital    : 10,799 features DOWNLOADED & CACHED in 2.4 s

Counting POIs per site per radius (Optimized sjoin)...
  poi_shop_250m: done  (sum=304)
  poi_shop_500m: done  (sum=1,266)
  poi_shop_1000m: done  (sum=5,750)
  poi_education_250m: done  (sum=24)
  poi_education_500m: done  (sum=146)
  poi_education_1000m: done  (sum=760)
  poi_hotel_250m: done  (sum=14)
  poi_hotel_500m: done  (sum=65)
  poi_hotel_1000m: done  (sum=252)
  poi_hospital_250m: done  (sum=26)
  poi_hospital_500m: done  (sum=111)
  poi_hospital_1000m: done  (sum=484)

All POI counts done in 0.8 s


## 8  Final Feature Table


In [19]:
KEEP = (
    # --- Site identifiers ---
    ['site_id', 'site_name', 'longitude', 'latitude',
     'municipality', 'district_code', 'road_code',
     'operator', 'interval_min', 'install_date']
    # --- Wegenregister road features ---
    + ['segment_id', 'road_name', 'morphology_en', 'road_category_code',
       'road_category_en', 'access_en', 'geometry_method_en',
       'manager_code', 'manager_label', 'length_m', 'dist_to_segment_m']
    # --- OSM bike features ---
    + ['bike_lane_width_m', 'has_cycleway', 'bike_lane_source']
    # --- OSM city distance ---
    + ['nearest_city', 'dist_to_city_m']
    # --- POI counts ---
    + [f'poi_{cat}_{r}m' for cat in POI_TAGS for r in POI_RADII]
    + ['geometry']
)

final_cols = [c for c in KEEP if c in sites_joined.columns]
sites_final = sites_joined[final_cols].copy()

print(f'Final enriched table: {sites_final.shape}')
display(sites_final.head(5))


Final enriched table: (151, 37)


,site_id,site_name,longitude,latitude,municipality,district_code,road_code,operator,interval_min,install_date,...,poi_education_250m,poi_education_500m,poi_education_1000m,poi_hotel_250m,poi_hotel_500m,poi_hotel_1000m,poi_hospital_250m,poi_hospital_500m,poi_hospital_1000m,geometry
0,100046096,Machelen,4.456122,50.916183,Machelen,AWV212,T2110002,Vlaamse Overheid A. Wegen enVerkeer,15,2019-08-22,...,0,0,0,0,0,0,0,0,1,POINT (156143.904 178432.685)
1,100052862,Brasschaat 2,4.471690,51.275120,Brasschaat,AWV123,N0010002,Vlaamse Overheid A. Wegen enVerkeer,15,2019-08-22,...,0,0,3,0,0,0,0,0,0,POINT (157182.957 218365.646)
2,100052863,Brasschaat 1,4.472220,51.275030,Brasschaat,AWV123,N0010001,Vlaamse Overheid A. Wegen enVerkeer,15,2019-08-22,...,0,0,3,0,0,0,0,0,0,POINT (157219.957 218355.684)
3,100052864,Balen 1,5.190110,51.160230,Balen,AWV114,N0180002,Vlaamse Overheid A. Wegen enVerkeer,15,2019-08-22,...,0,0,1,0,0,0,0,0,0,POINT (207457.483 205896.897)
4,100052865,Balen 2,5.190030,51.160180,Balen,AWV114,N0180002,Vlaamse Overheid A. Wegen enVerkeer,15,2019-08-22,...,0,0,1,0,0,0,0,0,0,POINT (207451.948 205891.273)


## 9  Data Quality Check


In [20]:
missing = sites_final.drop(columns='geometry').isna().sum()
pct     = (missing / len(sites_final) * 100).round(1)
dq = pd.DataFrame({'missing_n': missing, 'missing_%': pct})
print('Missing values per column:')
display(dq[dq['missing_n'] > 0].sort_values('missing_%', ascending=False))

# POI summary statistics
poi_cols = [c for c in sites_final.columns if c.startswith('poi_')]
print('\nPOI count summary:')
display(sites_final[poi_cols].describe().round(1))


Missing values per column:


,missing_n,missing_%
road_category_en,104,68.9
road_name,32,21.2
bike_lane_width_m,8,5.3
district_code,3,2.0
road_code,3,2.0
municipality,1,0.7



POI count summary:


,poi_shop_250m,poi_shop_500m,poi_shop_1000m,poi_education_250m,poi_education_500m,poi_education_1000m,poi_hotel_250m,poi_hotel_500m,poi_hotel_1000m,poi_hospital_250m,poi_hospital_500m,poi_hospital_1000m
count,151.0,151.0,151.0,151.0,151.0,151.0,151.0,151.0,151.0,151.0,151.0,151.0
mean,2.0,8.4,38.1,0.2,1.0,5.0,0.1,0.4,1.7,0.2,0.7,3.2
std,3.6,15.0,74.1,0.5,2.2,12.5,0.3,0.9,3.3,0.6,1.6,5.6
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
25%,0.0,2.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
50%,1.0,4.0,15.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,2.0
75%,3.0,8.0,36.0,0.0,1.0,4.0,0.0,1.0,2.0,0.0,1.0,4.0
max,22.0,138.0,462.0,5.0,13.0,108.0,2.0,6.0,19.0,3.0,10.0,33.0


In [26]:
sites_final[['bike_lane_width_m', 'bike_lane_source']].head()

,bike_lane_width_m,bike_lane_source
0,4.0,tier3:bike_highway_width
1,NaN,tier4:no_cycleway_tag
2,NaN,tier4:no_cycleway_tag
3,NaN,tier4:no_cycleway_tag
4,NaN,tier4:no_cycleway_tag


## 10  Export Enriched Data


In [ ]:
# CSV  (no geometry column – for quick inspection and modelling)
csv_out = os.path.join(OUTPUT_DIR, 'sites_enriched.csv')
sites_final.drop(columns='geometry').to_csv(csv_out, index=False)
print(f'CSV  saved : {csv_out}')

# GeoPackage (with geometry – for GIS / spatial modelling)
gpkg_out = os.path.join(OUTPUT_DIR, 'sites_enriched.gpkg')
sites_final.to_file(gpkg_out, driver='GPKG')
print(f'GPKG saved : {gpkg_out}')

print(f'\nDone. {len(sites_final)} sites x {len(sites_final.columns)} columns.')


## 11  Output Data Dictionary

| Column | Type | Source | Description |
|---|---|---|---|
| `site_id` | int | sites.csv | Unique traffic-count station identifier |
| `site_name` | str | sites.csv | Human-readable station name |
| `longitude` | float | sites.csv | WGS84 longitude |
| `latitude` | float | sites.csv | WGS84 latitude |
| `municipality` | str | sites.csv | Municipality name |
| `district_code` | str | sites.csv | AWV district code (e.g. AWV212) |
| `road_code` | str | sites.csv | Road identifier code (e.g. N0010002) |
| `operator` | str | sites.csv | Organisation operating the sensor |
| `interval_min` | int | sites.csv | Measurement interval in minutes |
| `install_date` | date | sites.csv | Date sensor was installed |
| `segment_id` | int | Wegenregister | Nearest road segment unique ID |
| `road_name` | str | Wegenregister | Street name from CRAB (left or right side) |
| `morphology_en` | str | Wegenregister | Road physical type (e.g. `single_carriageway`, `cycleway_footpath`) |
| `road_category_code` | str | Wegenregister | Functional category code (H, PI, PII, S, L, EW) |
| `road_category_en` | str | Wegenregister | Functional category in English |
| `access_en` | str | Wegenregister | Public accessibility (e.g. `public_road`, `private_road`) |
| `geometry_method_en` | str | Wegenregister | How geometry was captured (`surveyed` / `sketched`) |
| `manager_code` | str | Wegenregister | Road manager organisation code |
| `manager_label` | str | Wegenregister | Road manager full name |
| `length_m` | float | Wegenregister | Length of nearest road segment (metres) |
| `dist_to_segment_m` | float | Wegenregister | Distance from site point to nearest segment (metres) |
| `bike_lane_width_m` | float | OSM | Bike-lane width in metres; NaN if no numeric width tag found |
| `has_cycleway` | bool | OSM | True if any cycleway presence tag found within 200 m |
| `bike_lane_source` | str | OSM | Which tier/tag provided the width (for traceability) |
| `nearest_city` | str | OSM | Name of nearest OSM city/town node |
| `dist_to_city_m` | float | OSM | Haversine distance to nearest city/town centre (metres) |
| `poi_shop_250m` | int | OSM | Number of shops within 250 m |
| `poi_shop_500m` | int | OSM | Number of shops within 500 m |
| `poi_shop_1000m` | int | OSM | Number of shops within 1 000 m |
| `poi_education_250m` | int | OSM | Number of schools/universities within 250 m |
| `poi_education_500m` | int | OSM | Number of schools/universities within 500 m |
| `poi_education_1000m` | int | OSM | Number of schools/universities within 1 000 m |
| `poi_hotel_250m` | int | OSM | Number of hotels/hostels within 250 m |
| `poi_hotel_500m` | int | OSM | Number of hotels/hostels within 500 m |
| `poi_hotel_1000m` | int | OSM | Number of hotels/hostels within 1 000 m |
| `poi_hospital_250m` | int | OSM | Number of hospitals/clinics within 250 m |
| `poi_hospital_500m` | int | OSM | Number of hospitals/clinics within 500 m |
| `poi_hospital_1000m` | int | OSM | Number of hospitals/clinics within 1 000 m |
| `geometry` | Point | derived | Site point geometry in EPSG:31370 (Lambert 72) |


In [41]:
import geopandas as gpd
import pandas as pd

# 1. Define your file paths based on your folder structure
width_dbf_path = "../../data/extra/Wegenregister/Shapefile/AttWegbreedte.dbf"
roads_shp_path = "../../data/extra/Wegenregister/Shapefile/Wegsegment.shp"

# 2. Load the Width Attribute Table
print("Loading official Wegenregister widths...")
width_df = gpd.read_file(width_dbf_path)

# Let's peek at the columns to confirm the exact names 
# Usually, it's 'WS_OIDN' (the ID) and 'BREEDTE' (the width in meters)
print("Columns in width table:", width_df.columns.tolist())

# 3. Load your main road segments (if you don't already have them in memory)
print("Loading main road geometries...")
wegen_gdf = gpd.read_file(roads_shp_path)

# 4. Merge the width data into the main spatial dataset
print("Merging datasets on WS_OIDN...")
# We use a standard Pandas merge. We only select the ID and the Width column 
# from the DBF to avoid pulling in unnecessary metadata and bloating your RAM.
wegen_with_width = wegen_gdf.merge(
    width_df[['WS_OIDN', 'BREEDTE']], 
    on='WS_OIDN', 
    how='left'
)

# Rename it to English so it matches your XGBoost feature pipeline
wegen_with_width.rename(columns={'BREEDTE': 'official_road_width_m'}, inplace=True)

print("\n--- Merge Complete ---")
print(f"Total road segments: {len(wegen_with_width):,}")
print(f"Segments with official width data: {wegen_with_width['official_road_width_m'].notna().sum():,}")
print(wegen_with_width[['WS_OIDN', 'official_road_width_m']].head())

Loading official Wegenregister widths...
Columns in width table: ['WB_OIDN', 'WS_OIDN', 'WS_GIDN', 'BREEDTE', 'VANPOS', 'TOTPOS', 'BEGINTIJD', 'BEGINORG', 'LBLBGNORG']
Loading main road geometries...
Merging datasets on WS_OIDN...

--- Merge Complete ---
Total road segments: 862,547
Segments with official width data: 862,547
   WS_OIDN  official_road_width_m
0        1                     -8
1        4                      4
2        6                      4
3        7                     -8
4        8                      4
